# QCar cooperative perception, round by round

Replays the QCar dataset as if the two cars were driving it now, one **round** per frame, in temporal order:

1. **Ego, live:** its camera image goes through the encoder (LSS) and the BEV backbone, producing the feature it would transmit.
2. **Peer, received:** the peer is not run live. Its transmitted feature (the per-agent BEV feature right before fusion, which is what HEAL sends between agents) was precomputed once per method and is loaded from disk, as if it had just arrived over the network.
3. **Ego fuses:** it warps the peer's feature into its own frame with the pairwise pose, fuses, and the detection head outputs boxes.
4. **Scored against GT:** every round, plus running AP and recall over all rounds played so far. The same round also runs on the ego's feature alone, so you can see what cooperation adds.

**Everything is configured in `qcar/conf.json`** (`rounds_*` keys, plus `compare_zoo_dir`); this notebook hardcodes none of it:

| key | what it selects |
|---|---|
| `rounds_method` | fusion method from the zoo; its architecture and weights run |
| `rounds_stage_sources` | load `encoder` / `bev_backbone` / `head` weights from another zoo method (`null` = same as the fusion). Loads only if the tensors match, and warns: stages trained apart need alignment first (HEAL stage 2) |
| `rounds_split`, `rounds_scenarios` | which frames to play (`null` = every scenario of the split) |
| `rounds_iou` | IoU thresholds for the metrics (the first one is shown per round) |
| `rounds_cache_dir`, `rounds_cache_dtype` | where the peer features are stored; they are rebuilt automatically when stale (~1 min, ~0.5 GB per method) |
| `rounds_amp`, `rounds_device` | fp16 autocast (off = fp32) and the GPU |
| `rounds_play_interval_ms` | speed of the Play button |

The pipeline logic lives in `qcar/rounds.py`. It was verified to reproduce the zoo's `COMPARISON.json` AP on the full validation split.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

from qcar import config, rounds

conf = config.load_conf()
{k: v for k, v in conf.items() if k.startswith("rounds_")}

## Session

Builds the model (fusion method + its stages) and the dataset, and the peer feature cache if it is missing. Picking another fusion in the dropdown below rebuilds the session with that method's own trained weights.

In [ ]:
session = rounds.Rounds(conf)
print("method %s, stages from %s, %d rounds, peer cache %s"
      % (session.method, session.sources, len(session), session.cache_dir))

## Player

- **Play / pause**, or step with the slider and the ◀ ▶ buttons.
- **Top row:** each car's camera, with GT (yellow) and the cooperative prediction (magenta) projected into it.
- **Bottom row, in the ego's BEV frame (meters):**
  - the ego's own transmitted feature, with its ego-only prediction in cyan;
  - the peer's received feature, warped to the ego;
  - the fused feature;
  - the detection score map.
- The **running metrics** cover every round played so far; a round played twice counts once. **Reset metrics** starts over.

In [ ]:
fig = plt.figure(figsize=(17, 9.5))
plt.close(fig)  # drawn into `screen` only
screen = W.Output()

fusion = W.Dropdown(options=[m for m in rounds.zoo_methods(conf)], value=session.method,
                    description="fusion")
slider = W.IntSlider(0, 0, len(session) - 1, description="round", continuous_update=False,
                     layout=W.Layout(width="55%"))
play = W.Play(value=0, min=0, max=len(session) - 1, interval=conf["rounds_play_interval_ms"])
prev_btn, next_btn = W.Button(description="◀"), W.Button(description="▶")
reset_btn = W.Button(description="reset metrics")
status = W.Label()
W.jslink((play, "value"), (slider, "value"))


def show(r):
    result = session[r]
    rounds.draw(result, fig)
    with screen:
        screen.clear_output(wait=True)
        display(fig)


def on_fusion(change):
    global session
    status.value = "loading %s ..." % change["new"]
    session = rounds.Rounds(conf, change["new"])
    play.max = slider.max = len(session) - 1
    status.value = "%s: %d rounds" % (session.method, len(session))
    show(slider.value)


fusion.observe(on_fusion, names="value")
slider.observe(lambda c: show(c["new"]), names="value")
prev_btn.on_click(lambda _: setattr(slider, "value", max(slider.value - 1, 0)))
next_btn.on_click(lambda _: setattr(slider, "value", min(slider.value + 1, slider.max)))
reset_btn.on_click(lambda _: (session.reset_metrics(), show(slider.value)))

display(W.VBox([W.HBox([fusion, play, prev_btn, next_btn, slider, reset_btn]), status, screen]))
show(slider.value)

## Whole sequence without drawing

Runs every round of the session (about 0.3 s each on the RTX 3050) and prints the final running metrics: the same numbers as `qcar/compare.py`, from the round-by-round pipeline.

In [ ]:
session.reset_metrics()
for r in range(len(session)):
    result = session[r]
for mode in ("coop", "ego"):
    print(mode, {iou: {k: round(v, 3) for k, v in m.items()} for iou, m in result["running_metrics"][mode].items()})